# Verify lmz's GPU decoder on this card

**Set the runtime first:** Runtime → Change runtime type → **T4 GPU**.
Then Runtime → Run all. It takes under a minute.

## Why a T4 in particular

lmz's CUDA decoder has been *run* on one architecture, a Blackwell RTX 5080.
It is clean under `compute-sanitizer` and compiles from sm_75 to sm_121, but
counting `cp.async` instructions in the generated code says the untested
architectures are not equal:

| | `LDGSTS` (real `cp.async`) | what is unknown |
|---|---|---|
| **sm_75, Turing — this T4** | **0** | **correctness** |
| sm_80 / 86 / 89 | 38 | a throughput number |
| sm_90 / 120 | 41 | a throughput number |

Turing has no `cp.async` instruction, so the intrinsic falls back to a
synchronous copy. Everything at sm_80 and above runs the algorithm that was
already verified; **Turing runs different code that has never been executed.**
A Colab T4 is exactly that card, which is why this notebook exists.

Nothing here needs a data file or a login: the streams are built by lmz's own
encoder and checked against lmz's own decoder, so the oracle travels with the
question.


In [ ]:
!pip install -q lmzip
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv,noheader
!nvcc --version | tail -2


In [ ]:
!python -m lmz doctor --gpu-verify


## Please paste the block above into an issue

https://github.com/FanxinSun/lmz/issues

**A pass is evidence too** — right now there is one card's worth of it, and a
`verdict OK` from an sm_75 closes the only open correctness question. A
`MISMATCH` is more valuable still: it means the decoder correctly refused to
trust itself on hardware it had never seen, which is what it was built to do.

Either way lmz keeps working. The GPU decoder is optional in every direction:
no CUDA is installed by `pip install lmzip`, and a device that disagrees with
the CPU decoder is never used.
